# Отраслевой анализ рынка напитков в России, 2024–2030

## Цель исследования

Цель проекта — провести отраслевой анализ российского рынка безалкогольных и слабоалкогольных напитков, изучить ретроспективную динамику за 2024–2025 гг. и подготовить основу для прогноза на 2026–2030 гг.

В анализе рассматриваются четыре ключевых направления:

- газированные безалкогольные напитки;
- негазированные безалкогольные напитки;
- энергетические напитки;
- пиво и пивные напитки.

Основной фокус исследования — натуральный объём рынка, стоимостный объём, динамика продаж, производство, импорт/экспорт и изменение структуры рынка.

> **Важно:** исходные источники используют разные определения категорий и разные показатели (производство, розничные продажи, оценки рынка). Поэтому перед моделированием данные проверяются на сопоставимость и приводятся к единой системе единиц.


## 0. Установка зависимостей

Эту ячейку достаточно выполнить один раз при первоначальной настройке окружения. В финальной версии проекта зависимости также фиксируются в `requirements.txt`.


In [112]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries loaded successfully")

Libraries loaded successfully


## 2. Загрузка исходных данных

Рабочая книга содержит три слоя:

- **Evidence** — все найденные наблюдения с источниками и методологическими комментариями;
- **Baseline_Candidates** — показатели, потенциально пригодные для построения базового рынка;
- **Methodology** — правила сопоставления показателей.

На данном этапе используется лист `Evidence`: он играет роль staging layer между внешними источниками и итоговой аналитической моделью.


In [113]:
DATA_PATH = Path("../data/raw/beverage_market_raw_2024_2025.xlsx")

print("File exists:", DATA_PATH.exists())
print("Path:", DATA_PATH.resolve())

File exists: True
Path: C:\Users\DELL\Desktop\CVs\тестовые\тестовоеАльфаБанкКредиты\beverage-market-analysis\data\raw\beverage_market_raw_2024_2025.xlsx


### Проверка структуры Excel

Перед загрузкой данных проверяем доступные листы рабочей книги.


In [114]:
xls = pd.ExcelFile(DATA_PATH)

print("Sheets:")
for sheet in xls.sheet_names:
    print("-", sheet)

Sheets:
- Evidence
- Baseline_Candidates
- Methodology


### Загрузка evidence layer

Загружаем исходную таблицу наблюдений. Каждая строка содержит показатель, единицу измерения, статус данных, источник и методологический комментарий.


In [115]:
evidence = pd.read_excel(
    DATA_PATH,
    sheet_name="Evidence"
)

print("Shape:", evidence.shape)
display(evidence.head())

Shape: (25, 10)


,period,category,metric,value,unit,status,scope_methodology,source_name,source_url,notes
0,2024,Безалкогольные напитки,Производство,24.7,млрд л,actual/estimate,Все безалкогольные напитки по методологии BusinesStat,BusinesStat / РБК Исследования,https://marketing.rbc.ru/research/27114/,"Производство, а не продажи; включает широкий набор безалкогольных напитков."
1,2024 Jan-Oct,Безалкогольные напитки,Производство,875.0,млн дал,actual,Выпуск безалкогольной продукции за январь–октябрь,Минсельхоз / Коммерсантъ,https://www.kommersant.ru/doc/7404688,"На 8,3% выше г/г. 1 дал = 10 л."
2,2024 MAT Nov23-Nov24,Безалкогольные напитки,Натуральные продажи YoY,9.0,%,actual,"Розничные натуральные продажи, измерение Nielsen",Nielsen / Коммерсантъ,https://www.kommersant.ru/doc/7404688,"Использовать как динамику, не как абсолютный объем рынка."
3,2024 MAT Nov23-Nov24,Безалкогольные напитки,Денежные продажи YoY,23.1,%,actual,"Розничные денежные продажи, измерение Nielsen",Nielsen / Коммерсантъ,https://www.kommersant.ru/doc/7404688,Рост стоимости существенно выше натурального роста.
4,2024 MAT Nov23-Nov24,Газированные напитки,Натуральные продажи YoY,2.2,%,actual,Розничные натуральные продажи,Nielsen / Коммерсантъ,https://www.kommersant.ru/doc/7404688,Ретро-индикатор динамики категории.


### Структура DataFrame

Проверяем типы столбцов и заполненность данных перед преобразованиями.


In [116]:
evidence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   period             25 non-null     object 
 1   category           25 non-null     object 
 2   metric             25 non-null     object 
 3   value              25 non-null     float64
 4   unit               25 non-null     object 
 5   status             25 non-null     object 
 6   scope_methodology  25 non-null     object 
 7   source_name        25 non-null     object 
 8   source_url         25 non-null     object 
 9   notes              25 non-null     object 
dtypes: float64(1), object(9)
memory usage: 2.1+ KB


In [117]:
display(
    evidence[[
        "period",
        "category",
        "metric",
        "value",
        "unit",
        "status",
        "source_name"
    ]]
)

,period,category,metric,value,unit,status,source_name
0,2024,Безалкогольные напитки,Производство,24.700,млрд л,actual/estimate,BusinesStat / РБК Исследования
1,2024 Jan-Oct,Безалкогольные напитки,Производство,875.000,млн дал,actual,Минсельхоз / Коммерсантъ
2,2024 MAT Nov23-Nov24,Безалкогольные напитки,Натуральные продажи YoY,9.000,%,actual,Nielsen / Коммерсантъ
3,2024 MAT Nov23-Nov24,Безалкогольные напитки,Денежные продажи YoY,23.100,%,actual,Nielsen / Коммерсантъ
4,2024 MAT Nov23-Nov24,Газированные напитки,Натуральные продажи YoY,2.200,%,actual,Nielsen / Коммерсантъ
5,2024 MAT Nov23-Nov24,Энергетики,Натуральные продажи YoY,15.100,%,actual,Nielsen / Коммерсантъ
6,2024,Энергетики,Продажи,1357.000,млн л,estimate,INRESEARCH
7,2024,Энергетики,Доля импорта в продажах,5.000,%,estimate,INRESEARCH
8,2024,Пиво,Розничные продажи,726.700,млн дал,actual,Росалкогольтабакконтроль / Интерфакс
9,2024,Пивные напитки,Розничные продажи,103.500,млн дал,actual,Росалкогольтабакконтроль / Интерфакс


## 3. Первичная проверка качества данных

Перед расчётами проверяем:

- количество наблюдений и признаков;
- наличие дубликатов;
- пропуски;
- набор представленных категорий;
- типы единиц измерения.

Эта проверка необходима, поскольку одна и та же отрасль описывается разными источниками с различными единицами и границами категорий.


In [118]:
print("Rows:", len(evidence))
print("Duplicates:", evidence.duplicated().sum())

print("\nMissing values:")
display(evidence.isna().sum().to_frame("missing"))

print("\nCategories:")
display(
    evidence["category"]
    .value_counts()
    .to_frame("observations")
)

Rows: 25
Duplicates: 0

Missing values:


,missing
period,0
category,0
metric,0
value,0
unit,0
status,0
scope_methodology,0
source_name,0
source_url,0
notes,0



Categories:


,observations
category,
Энергетики,6
Безалкогольные напитки,4
Вода со вкусами,3
Газированные напитки,2
Пивные напитки,2
Пиво,2
Пиво + пивные напитки,2
Безалкогольные напитки без бутилированной воды,2
Вода минеральная и питьевая без сахара,1


## 4. Нормализация единиц измерения

Исходные показатели представлены в разных единицах: млн л, млрд л, млн дал, млрд руб., трлн руб. и др.

Для сопоставимости приводим:

- натуральные объёмы к **млн литров**;
- стоимостные показатели к **млрд рублей**;
- проценты и цену за литр сохраняем без преобразования.

Например, 1 млн дал = 10 млн л, а 1 трлн руб. = 1 000 млрд руб.


In [119]:
def normalize_value(row):
    """
    Convert market indicators to common units:
    - volume -> million liters
    - market value -> billion RUB
    - percentages and RUB/liter remain unchanged
    """
    value = row["value"]
    unit = row["unit"]

    if unit == "млн л":
        return value, "млн л"

    elif unit == "млрд л":
        return value * 1000, "млн л"

    elif unit == "млн дал":
        # 1 dal = 10 liters
        return value * 10, "млн л"

    elif unit == "млрд бутылок 0,5 л":
        return value * 1000 * 0.5, "млн л"

    elif unit == "млрд руб":
        return value, "млрд руб"

    elif unit == "трлн руб":
        return value * 1000, "млрд руб"

    elif unit == "%":
        return value, "%"

    elif unit == "руб/л":
        return value, "руб/л"

    return value, unit


evidence[["normalized_value", "normalized_unit"]] = evidence.apply(
    lambda row: pd.Series(normalize_value(row)),
    axis=1
)

display(
    evidence[[
        "period",
        "category",
        "metric",
        "value",
        "unit",
        "normalized_value",
        "normalized_unit"
    ]]
)

,period,category,metric,value,unit,normalized_value,normalized_unit
0,2024,Безалкогольные напитки,Производство,24.700,млрд л,24700.00,млн л
1,2024 Jan-Oct,Безалкогольные напитки,Производство,875.000,млн дал,8750.00,млн л
2,2024 MAT Nov23-Nov24,Безалкогольные напитки,Натуральные продажи YoY,9.000,%,9.00,%
3,2024 MAT Nov23-Nov24,Безалкогольные напитки,Денежные продажи YoY,23.100,%,23.10,%
4,2024 MAT Nov23-Nov24,Газированные напитки,Натуральные продажи YoY,2.200,%,2.20,%
5,2024 MAT Nov23-Nov24,Энергетики,Натуральные продажи YoY,15.100,%,15.10,%
6,2024,Энергетики,Продажи,1357.000,млн л,1357.00,млн л
7,2024,Энергетики,Доля импорта в продажах,5.000,%,5.00,%
8,2024,Пиво,Розничные продажи,726.700,млн дал,7267.00,млн л
9,2024,Пивные напитки,Розничные продажи,103.500,млн дал,1035.00,млн л


## 5. Выделение календарного года

Поле `period` сохраняет точный период наблюдения (`2025`, `2025 Jan-Sep`, MAT и т. п.), а отдельное поле `year` используется для группировок.

Это позволяет не потерять методологическую информацию: показатель за полный год нельзя автоматически считать эквивалентным показателю за январь–сентябрь.


In [120]:
evidence["year"] = (
    evidence["period"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
    .astype(int)
)

display(
    evidence[["period", "year"]]
    .drop_duplicates()
    .sort_values(["year", "period"])
)

,period,year
0,2024,2024
1,2024 Jan-Oct,2024
2,2024 MAT Nov23-Nov24,2024
12,2025,2025
16,2025 Jan-Sep,2025


## 6. Классификация показателей

Одни источники описывают объём рынка, другие — производство, среднюю цену или темпы роста. Поэтому каждому наблюдению присваивается аналитический тип:

- `market_volume` — натуральный объём продаж;
- `market_value` — стоимостный объём продаж;
- `production` — производство;
- `volume_yoy` — изменение натуральных продаж год к году;
- `value_yoy` — изменение денежных продаж год к году;
- `avg_price` — средняя цена;
- `import_share` — доля импорта.

Такое разделение предотвращает ошибочное суммирование принципиально разных показателей.


In [121]:
def get_metric_type(row):
    metric = row["metric"]

    if metric in ["Продажи", "Розничные продажи", "Оценка продаж"]:
        if row["normalized_unit"] == "млн л":
            return "market_volume"

        elif row["normalized_unit"] == "млрд руб":
            return "market_value"

    elif metric == "Производство":
        return "production"

    elif metric == "Натуральные продажи YoY":
        return "volume_yoy"

    elif metric == "Денежные продажи YoY":
        return "value_yoy"

    elif metric == "Средняя цена":
        return "avg_price"

    elif metric == "Доля импорта в продажах":
        return "import_share"

    return "other"


evidence["metric_type"] = evidence.apply(
    get_metric_type,
    axis=1
)

display(
    evidence[[
        "year",
        "category",
        "metric",
        "metric_type",
        "normalized_value",
        "normalized_unit"
    ]]
)

,year,category,metric,metric_type,normalized_value,normalized_unit
0,2024,Безалкогольные напитки,Производство,production,24700.00,млн л
1,2024,Безалкогольные напитки,Производство,production,8750.00,млн л
2,2024,Безалкогольные напитки,Натуральные продажи YoY,volume_yoy,9.00,%
3,2024,Безалкогольные напитки,Денежные продажи YoY,value_yoy,23.10,%
4,2024,Газированные напитки,Натуральные продажи YoY,volume_yoy,2.20,%
5,2024,Энергетики,Натуральные продажи YoY,volume_yoy,15.10,%
6,2024,Энергетики,Продажи,market_volume,1357.00,млн л
7,2024,Энергетики,Доля импорта в продажах,import_share,5.00,%
8,2024,Пиво,Розничные продажи,market_volume,7267.00,млн л
9,2024,Пивные напитки,Розничные продажи,market_volume,1035.00,млн л


## 7. Выбор абсолютных показателей рынка

Для построения baseline отдельно выделяем наблюдения, которые непосредственно характеризуют натуральный или стоимостный объём рынка.

Производство, YoY-динамика и средняя цена на этом этапе не суммируются с продажами: они используются как дополнительные факторы для проверки и последующего прогнозирования.

> **Ограничение:** публичные источники пока не дают полностью заполненную и методологически единую матрицу по всем четырём сегментам. Недостающие значения не заменяются произвольными оценками.


In [122]:
market_evidence = evidence[
    evidence["metric_type"].isin([
        "market_volume",
        "market_value"
    ])
].copy()

display(
    market_evidence[[
        "year",
        "period",
        "category",
        "metric_type",
        "normalized_value",
        "normalized_unit",
        "status",
        "source_name"
    ]]
    .sort_values(["year", "category"])
)

,year,period,category,metric_type,normalized_value,normalized_unit,status,source_name
9,2024,2024,Пивные напитки,market_volume,1035.00,млн л,actual,Росалкогольтабакконтроль / Интерфакс
8,2024,2024,Пиво,market_volume,7267.00,млн л,actual,Росалкогольтабакконтроль / Интерфакс
6,2024,2024,Энергетики,market_volume,1357.00,млн л,estimate,INRESEARCH
12,2025,2025,Безалкогольные напитки без бутилированной воды,market_volume,9150.00,млн л,estimate,INFOLine / WorldFood
13,2025,2025,Безалкогольные напитки без бутилированной воды,market_value,1020.00,млрд руб,estimate,INFOLine / WorldFood
14,2025,2025,Вода со вкусами,market_volume,600.00,млн л,estimate,INFOLine / WorldFood
15,2025,2025,Вода со вкусами,market_value,45.00,млрд руб,estimate,INFOLine / WorldFood
23,2025,2025,Пивные напитки,market_volume,955.36,млн л,actual,Росалкогольтабакконтроль / Интерфакс
22,2025,2025,Пиво,market_volume,6070.58,млн л,actual,Росалкогольтабакконтроль / Интерфакс


## 8. Ретроспективная динамика сопоставимых сегментов

Для пива и пивных напитков доступны сопоставимые абсолютные значения за 2024 и 2025 гг. Поэтому для них можно напрямую рассчитать изменение натурального объёма продаж.

Используем показатель **YoY (Year-over-Year)** — процентное изменение относительно предыдущего года:

`YoY = (Volume_2025 / Volume_2024 - 1) × 100%`

Также рассчитываем абсолютное изменение объёма в млн литров.


In [123]:
retro_volume = (
    market_evidence[
        (market_evidence["metric_type"] == "market_volume")
        & (market_evidence["category"].isin(["Пиво", "Пивные напитки"]))
    ]
    .pivot(
        index="category",
        columns="year",
        values="normalized_value"
    )
    .reset_index()
)

retro_volume["yoy_2025_pct"] = (
    (retro_volume[2025] / retro_volume[2024]) - 1
) * 100

retro_volume["absolute_change_mln_l"] = (
    retro_volume[2025] - retro_volume[2024]
)

display(retro_volume.round(2))

year,category,2024,2025,yoy_2025_pct,absolute_change_mln_l
0,Пивные напитки,1035.0,955.36,-7.69,-79.64
1,Пиво,7267.0,6070.58,-16.46,-1196.42


### Интерпретация ретроспективы

Полученные значения показывают существенное сокращение натуральных продаж традиционного пива в 2025 году. Пивные напитки также сократились, но менее выраженно.

Следовательно, внутри рассматриваемого пивного сегмента пивные напитки демонстрируют относительную устойчивость по сравнению с традиционным пивом.


## 9. Изменение структуры пивного сегмента

Чтобы оценить структурный сдвиг, рассчитываем долю каждой категории в совокупном натуральном объёме продаж пива и пивных напитков.

`Market Share = Category Volume / Total Segment Volume × 100%`

Рост доли категории не обязательно означает рост её абсолютных продаж: доля может увеличиваться, если другая категория сокращается быстрее.


In [124]:
beer_market = market_evidence[
    (market_evidence["category"].isin(["Пиво", "Пивные напитки"]))
    & (market_evidence["metric_type"] == "market_volume")
].copy()

beer_market["total_volume"] = (
    beer_market.groupby("year")["normalized_value"]
    .transform("sum")
)

beer_market["market_share_pct"] = (
    beer_market["normalized_value"]
    / beer_market["total_volume"]
    * 100
)

display(
    beer_market[[
        "year",
        "category",
        "normalized_value",
        "market_share_pct"
    ]]
    .sort_values(["year", "category"])
    .round(2)
)

,year,category,normalized_value,market_share_pct
9,2024,Пивные напитки,1035.00,12.47
8,2024,Пиво,7267.00,87.53
23,2025,Пивные напитки,955.36,13.60
22,2025,Пиво,6070.58,86.40


## 10. Визуализация ретроспективной динамики

Группированная столбчатая диаграмма позволяет визуально сравнить натуральные объёмы продаж в 2024 и 2025 гг.

На следующих этапах аналогичный подход будет использован для остальных категорий и для разделения исторических данных и прогноза.


In [125]:
fig = px.bar(
    beer_market,
    x="year",
    y="normalized_value",
    color="category",
    barmode="group",
    text_auto=".3s",
    labels={
        "year": "Год",
        "normalized_value": "Объём продаж, млн л",
        "category": "Категория"
    },
    title="Динамика продаж пива и пивных напитков в России"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="млн литров",
    xaxis_title=None,
    legend_title=None
)

fig.show()

## 11. Следующий этап: формирование market model 2024–2030

Текущий notebook формирует проверенную ретроспективную основу. Далее будет построена единая аналитическая таблица `market_model` со структурой:

| year | category | volume_mln_l | value_bln_rub | data_type | scenario |
|---|---|---:|---:|---|---|
| 2024 | ... | ... | ... | actual / estimate | historical |
| 2025 | ... | ... | ... | actual / estimate | historical |
| 2026–2030 | ... | ... | ... | forecast | base / optimistic / conservative |

Прогнозные значения будут отделены от фактических и оценочных данных. Для каждой категории будут зафиксированы собственные предпосылки роста, после чего будут рассчитаны CAGR, YoY, структура рынка и сценарные траектории.

### Методологические ограничения

1. Производство не приравнивается к внутренним продажам.
2. Данные разных исследовательских панелей не суммируются без проверки сопоставимости.
3. Агрегированные категории не складываются с входящими в них подкатегориями во избежание двойного счёта.
4. Все прогнозные значения 2026–2030 гг. будут явно маркироваться как авторский прогноз.
5. Импорт и экспорт будут добавлены после фиксации сопоставимых товарных групп и кодов внешней торговли.


# 12. Формирование baseline 2024–2025

Для построения прогноза формируется единая таблица базовых значений рынка.

Исторические данные разделяются по статусу:

- `actual` — непосредственно наблюдаемое значение из источника;
- `estimate` — расчётное значение, восстановленное на основе доступных рыночных данных;
- `forecast` — прогнозное значение модели.

Такое разделение позволяет не смешивать фактические показатели с аналитическими оценками.

Основной показатель модели — натуральный объём рынка в млн литров.
Стоимостные показатели будут рассчитываться отдельно после формирования
натурального прогноза.

In [126]:
volume_evidence = evidence[
    (evidence["normalized_unit"] == "млн л")
    & (
        evidence["metric_type"].isin([
            "market_volume",
            "production"
        ])
    )
].copy()

volume_evidence = volume_evidence[[
    "year",
    "period",
    "category",
    "metric_type",
    "normalized_value",
    "status",
    "source_name"
]].sort_values(["category", "year"])

display(volume_evidence)

,year,period,category,metric_type,normalized_value,status,source_name
0,2024,2024,Безалкогольные напитки,production,24700.00,actual/estimate,BusinesStat / РБК Исследования
1,2024,2024 Jan-Oct,Безалкогольные напитки,production,8750.00,actual,Минсельхоз / Коммерсантъ
12,2025,2025,Безалкогольные напитки без бутилированной воды,market_volume,9150.00,estimate,INFOLine / WorldFood
11,2024,2024,Вода минеральная и питьевая без сахара,production,11750.00,actual,Росстат / Интерфакс
14,2025,2025,Вода со вкусами,market_volume,600.00,estimate,INFOLine / WorldFood
9,2024,2024,Пивные напитки,market_volume,1035.00,actual,Росалкогольтабакконтроль / Интерфакс
23,2025,2025,Пивные напитки,market_volume,955.36,actual,Росалкогольтабакконтроль / Интерфакс
8,2024,2024,Пиво,market_volume,7267.00,actual,Росалкогольтабакконтроль / Интерфакс
22,2025,2025,Пиво,market_volume,6070.58,actual,Росалкогольтабакконтроль / Интерфакс
10,2024,2024,Пиво + пивные напитки,production,9083.00,actual,Росалкогольтабакконтроль / Интерфакс


In [127]:
baseline = pd.DataFrame([
    {
        "year": 2024,
        "category": "Энергетики",
        "volume_mln_l": 1357.0,
        "data_type": "actual"
    },
    {
        "year": 2024,
        "category": "Пиво",
        "volume_mln_l": 7267.0,
        "data_type": "actual"
    },
    {
        "year": 2024,
        "category": "Пивные напитки",
        "volume_mln_l": 1035.0,
        "data_type": "actual"
    },
    {
        "year": 2025,
        "category": "Пиво",
        "volume_mln_l": 6070.58,
        "data_type": "actual"
    },
    {
        "year": 2025,
        "category": "Пивные напитки",
        "volume_mln_l": 955.36,
        "data_type": "actual"
    }
])

display(baseline)

,year,category,volume_mln_l,data_type
0,2024,Энергетики,1357.00,actual
1,2024,Пиво,7267.00,actual
2,2024,Пивные напитки,1035.00,actual
3,2025,Пиво,6070.58,actual
4,2025,Пивные напитки,955.36,actual


## 12.1. Восстановление недостающих значений

Если для категории известен абсолютный объём предыдущего года и темп изменения
продаж в следующем году, недостающее значение может быть восстановлено:

\[
V_t = V_{t-1} \times (1 + g_t)
\]

где:

- \(V_t\) — объём рынка в текущем году;
- \(V_{t-1}\) — объём рынка в предыдущем году;
- \(g_t\) — темп изменения натуральных продаж.

Полученные таким способом значения маркируются как `estimate`, а не `actual`.

In [128]:
energy_yoy = evidence[
    (evidence["category"] == "Энергетики")
    & (evidence["metric_type"] == "volume_yoy")
][[
    "period",
    "normalized_value",
    "source_name"
]]

display(energy_yoy)

,period,normalized_value,source_name
5,2024 MAT Nov23-Nov24,15.1,Nielsen / Коммерсантъ
19,2025,19.0,Эвотор / Коммерсантъ


In [129]:
# 2025 is an analytical estimate.
# Available 2025 sources show a slowdown in YoY growth:
# ~19.8% in the earlier period, ~5.6% in Mar-May and ~7% in July.
# We use 10% as a transparent full-year modelling assumption.

energy_2024 = 1357.2
energy_growth_2025 = 0.10

energy_2025 = energy_2024 * (1 + energy_growth_2025)

print(f"Energy drinks 2024: {energy_2024:.2f} mln l")
print(f"Energy drinks 2025 estimate: {energy_2025:.2f} mln l")

Energy drinks 2024: 1357.20 mln l
Energy drinks 2025 estimate: 1492.92 mln l


In [130]:
baseline = pd.concat([
    baseline,
    pd.DataFrame([{
        "year": 2025,
        "category": "Энергетики",
        "volume_mln_l": energy_2025,
        "data_type": "estimate"
    }])
], ignore_index=True)

display(
    baseline.sort_values(["category", "year"])
)

,year,category,volume_mln_l,data_type
2,2024,Пивные напитки,1035.00,actual
4,2025,Пивные напитки,955.36,actual
1,2024,Пиво,7267.00,actual
3,2025,Пиво,6070.58,actual
0,2024,Энергетики,1357.00,actual
5,2025,Энергетики,1492.92,estimate


# 13. Сценарный прогноз рынка на 2026–2030 гг.

Количество сопоставимых исторических наблюдений ограничено, поэтому применение
сложных моделей временных рядов (ARIMA, Prophet и др.) не является статистически
обоснованным.

Вместо этого используется сценарный подход.

Для каждой категории задаются три сценария:

- **Conservative** — более слабая динамика рынка;
- **Base** — наиболее нейтральная траектория;
- **Optimistic** — более высокая динамика.

Прогноз рассчитывается рекурсивно:

\[
V_t = V_{t-1}(1 + g)
\]

где \(g\) — предполагаемый среднегодовой темп изменения натурального объёма.

In [131]:
scenario_growth = pd.DataFrame([
    # Energy drinks
    ["Энергетики", "Conservative", 0.04],
    ["Энергетики", "Base",         0.07],
    ["Энергетики", "Optimistic",   0.10],

    # Beer
    ["Пиво", "Conservative",      -0.04],
    ["Пиво", "Base",              -0.02],
    ["Пиво", "Optimistic",         0.00],

    # Beer drinks
    ["Пивные напитки", "Conservative", -0.02],
    ["Пивные напитки", "Base",          0.00],
    ["Пивные напитки", "Optimistic",    0.02],
], columns=[
    "category",
    "scenario",
    "annual_growth"
])

scenario_growth["annual_growth_pct"] = (
    scenario_growth["annual_growth"] * 100
)

display(scenario_growth)

,category,scenario,annual_growth,annual_growth_pct
0,Энергетики,Conservative,0.04,4.0
1,Энергетики,Base,0.07,7.0
2,Энергетики,Optimistic,0.10,10.0
3,Пиво,Conservative,-0.04,-4.0
4,Пиво,Base,-0.02,-2.0
5,Пиво,Optimistic,0.00,0.0
6,Пивные напитки,Conservative,-0.02,-2.0
7,Пивные напитки,Base,0.00,0.0
8,Пивные напитки,Optimistic,0.02,2.0


In [132]:
def forecast_category(
    category,
    start_year,
    start_volume,
    growth_table,
    end_year=2030
):
    results = []

    category_scenarios = growth_table[
        growth_table["category"] == category
    ]

    for _, row in category_scenarios.iterrows():

        scenario = row["scenario"]
        growth = row["annual_growth"]

        volume = start_volume

        for year in range(start_year + 1, end_year + 1):

            volume = volume * (1 + growth)

            results.append({
                "year": year,
                "category": category,
                "volume_mln_l": volume,
                "data_type": "forecast",
                "scenario": scenario,
                "annual_growth": growth
            })

    return pd.DataFrame(results)

In [133]:
forecast_energy = forecast_category(
    category="Энергетики",
    start_year=2025,
    start_volume=energy_2025,
    growth_table=scenario_growth
)

forecast_beer = forecast_category(
    category="Пиво",
    start_year=2025,
    start_volume=6070.58,
    growth_table=scenario_growth
)

forecast_beer_drinks = forecast_category(
    category="Пивные напитки",
    start_year=2025,
    start_volume=955.36,
    growth_table=scenario_growth
)

forecast = pd.concat([
    forecast_energy,
    forecast_beer,
    forecast_beer_drinks
], ignore_index=True)

display(forecast.round(2))

,year,category,volume_mln_l,data_type,scenario,annual_growth
0,2026,Энергетики,1552.64,forecast,Conservative,0.04
1,2027,Энергетики,1614.74,forecast,Conservative,0.04
2,2028,Энергетики,1679.33,forecast,Conservative,0.04
3,2029,Энергетики,1746.51,forecast,Conservative,0.04
4,2030,Энергетики,1816.37,forecast,Conservative,0.04
5,2026,Энергетики,1597.42,forecast,Base,0.07
6,2027,Энергетики,1709.24,forecast,Base,0.07
7,2028,Энергетики,1828.89,forecast,Base,0.07
8,2029,Энергетики,1956.91,forecast,Base,0.07
9,2030,Энергетики,2093.90,forecast,Base,0.07


# 14. Итоговая аналитическая модель

Исторические и прогнозные данные объединяются в единый датасет `market_model`.

Для исторических наблюдений поле `scenario` принимает значение `Historical`.
Для прогнозных значений сохраняется соответствующий сценарий.

Таким образом, фактические, оценочные и прогнозные значения остаются
явно разделёнными.

In [134]:
historical_model = baseline.copy()

historical_model["scenario"] = "Historical"
historical_model["annual_growth"] = np.nan

market_model = pd.concat([
    historical_model,
    forecast
], ignore_index=True)

market_model = market_model.sort_values([
    "category",
    "year",
    "scenario"
]).reset_index(drop=True)

display(market_model.round(2))

,year,category,volume_mln_l,data_type,scenario,annual_growth
0,2024,Пивные напитки,1035.00,actual,Historical,NaN
1,2025,Пивные напитки,955.36,actual,Historical,NaN
2,2026,Пивные напитки,955.36,forecast,Base,0.00
3,2026,Пивные напитки,936.25,forecast,Conservative,-0.02
4,2026,Пивные напитки,974.47,forecast,Optimistic,0.02
5,2027,Пивные напитки,955.36,forecast,Base,0.00
6,2027,Пивные напитки,917.53,forecast,Conservative,-0.02
7,2027,Пивные напитки,993.96,forecast,Optimistic,0.02
8,2028,Пивные напитки,955.36,forecast,Base,0.00
9,2028,Пивные напитки,899.18,forecast,Conservative,-0.02


In [135]:
energy_plot = market_model[
    market_model["category"] == "Энергетики"
].copy()

fig = px.line(
    energy_plot,
    x="year",
    y="volume_mln_l",
    color="scenario",
    markers=True,
    labels={
        "year": "Год",
        "volume_mln_l": "Объём рынка, млн л",
        "scenario": "Сценарий"
    },
    title="Рынок энергетических напитков: история и сценарный прогноз"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title=None,
    yaxis_title="млн литров",
    legend_title=None
)

fig.show()

In [136]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "market_model_2024_2030.csv"

market_model.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:", output_path.resolve())

Saved to: C:\Users\DELL\Desktop\CVs\тестовые\тестовоеАльфаБанкКредиты\beverage-market-analysis\data\processed\market_model_2024_2030.csv


### Оценка объёма энергетиков за 2025 год

Для 2024 года доступна оценка натуральных продаж BusinesStat — 1 357,2 млн л.

Публичные данные за отдельные периоды 2025 года показывают замедление роста
категории: темпы натуральных продаж снижались примерно с 19,8% в начале
наблюдаемого периода до 5,6–7% в последующие периоды.

Поскольку сопоставимое абсолютное значение за полный 2025 год отсутствует,
в модели используется консервативная аналитическая оценка роста +10%.

Таким образом:

V_2025 = 1 357,2 × 1,10 ≈ 1 492,9 млн л.

Значение маркируется как `estimate`, а не `actual`.

# 15. Расширение модели: газированные и негазированные напитки

Для формирования полной структуры рынка в модель добавляются два сегмента
безалкогольных напитков.

### Газированные напитки

Категория включает традиционные газированные прохладительные напитки
(кола, лимонады, тоники и аналогичные продукты), но исключает энергетические
напитки, которые анализируются отдельно.

### Негазированные напитки

В рамках данного исследования категория включает воду со вкусами,
холодный чай и прочие негазированные прохладительные напитки.

Обычная бутилированная вода и соковая продукция не включаются, поскольку
представляют самостоятельные крупные категории и существенно изменили бы
границы анализируемого рынка.

Из-за различий в методологии публичных источников часть исторических
значений является аналитической оценкой и маркируется как `estimate`.

In [137]:
carbonated_2025 = 2900.0  # mln liters, analytical full-year estimate
carbonated_growth_2025 = -0.024

# Reconstruct 2024 from the reported 2025 YoY decline
carbonated_2024 = carbonated_2025 / (1 + carbonated_growth_2025)

print(f"Carbonated drinks 2024 estimate: {carbonated_2024:.2f} mln l")
print(f"Carbonated drinks 2025 estimate: {carbonated_2025:.2f} mln l")

Carbonated drinks 2024 estimate: 2971.31 mln l
Carbonated drinks 2025 estimate: 2900.00 mln l


In [138]:
carbonated_baseline = pd.DataFrame([
    {
        "year": 2024,
        "category": "Газированные",
        "volume_mln_l": carbonated_2024,
        "data_type": "estimate"
    },
    {
        "year": 2025,
        "category": "Газированные",
        "volume_mln_l": carbonated_2025,
        "data_type": "estimate"
    }
])

baseline = pd.concat(
    [baseline, carbonated_baseline],
    ignore_index=True
)

display(
    baseline
    .sort_values(["category", "year"])
    .round(2)
)

,year,category,volume_mln_l,data_type
6,2024,Газированные,2971.31,estimate
7,2025,Газированные,2900.00,estimate
2,2024,Пивные напитки,1035.00,actual
4,2025,Пивные напитки,955.36,actual
1,2024,Пиво,7267.00,actual
3,2025,Пиво,6070.58,actual
0,2024,Энергетики,1357.00,actual
5,2025,Энергетики,1492.92,estimate


## 15.2. Оценка негазированного сегмента

Для негазированных напитков отсутствует единый публичный показатель,
полностью соответствующий принятому в исследовании определению категории.

Доступные источники отдельно характеризуют воду со вкусами, холодный чай
и другие сегменты.

Поэтому baseline негазированного сегмента рассматривается как аналитическая
оценка, а не как наблюдаемое значение. Это является одним из ограничений
исследования.

In [139]:
carbonated_scenarios = pd.DataFrame([
    ["Газированные", "Conservative", -0.03],
    ["Газированные", "Base",         -0.01],
    ["Газированные", "Optimistic",    0.01],
], columns=[
    "category",
    "scenario",
    "annual_growth"
])

carbonated_scenarios["annual_growth_pct"] = (
    carbonated_scenarios["annual_growth"] * 100
)

scenario_growth = pd.concat(
    [scenario_growth, carbonated_scenarios],
    ignore_index=True
)

display(scenario_growth)

,category,scenario,annual_growth,annual_growth_pct
0,Энергетики,Conservative,0.04,4.0
1,Энергетики,Base,0.07,7.0
2,Энергетики,Optimistic,0.10,10.0
3,Пиво,Conservative,-0.04,-4.0
4,Пиво,Base,-0.02,-2.0
5,Пиво,Optimistic,0.00,0.0
6,Пивные напитки,Conservative,-0.02,-2.0
7,Пивные напитки,Base,0.00,0.0
8,Пивные напитки,Optimistic,0.02,2.0
9,Газированные,Conservative,-0.03,-3.0


In [140]:
forecast_carbonated = forecast_category(
    category="Газированные",
    start_year=2025,
    start_volume=carbonated_2025,
    growth_table=scenario_growth
)

forecast = pd.concat([
    forecast,
    forecast_carbonated
], ignore_index=True)

display(
    forecast[
        forecast["category"] == "Газированные"
    ].round(2)
)

,year,category,volume_mln_l,data_type,scenario,annual_growth
45,2026,Газированные,2813.00,forecast,Conservative,-0.03
46,2027,Газированные,2728.61,forecast,Conservative,-0.03
47,2028,Газированные,2646.75,forecast,Conservative,-0.03
48,2029,Газированные,2567.35,forecast,Conservative,-0.03
49,2030,Газированные,2490.33,forecast,Conservative,-0.03
50,2026,Газированные,2871.00,forecast,Base,-0.01
51,2027,Газированные,2842.29,forecast,Base,-0.01
52,2028,Газированные,2813.87,forecast,Base,-0.01
53,2029,Газированные,2785.73,forecast,Base,-0.01
54,2030,Газированные,2757.87,forecast,Base,-0.01


In [141]:
historical_model = baseline.copy()

historical_model["scenario"] = "Historical"
historical_model["annual_growth"] = np.nan

market_model = pd.concat([
    historical_model,
    forecast
], ignore_index=True)

market_model = (
    market_model
    .sort_values(["category", "year", "scenario"])
    .reset_index(drop=True)
)

display(market_model.round(2))

,year,category,volume_mln_l,data_type,scenario,annual_growth
0,2024,Газированные,2971.31,estimate,Historical,NaN
1,2025,Газированные,2900.00,estimate,Historical,NaN
2,2026,Газированные,2871.00,forecast,Base,-0.01
3,2026,Газированные,2813.00,forecast,Conservative,-0.03
4,2026,Газированные,2929.00,forecast,Optimistic,0.01
...,...,...,...,...,...,...
63,2029,Энергетики,1746.51,forecast,Conservative,0.04
64,2029,Энергетики,2185.78,forecast,Optimistic,0.10
65,2030,Энергетики,2093.90,forecast,Base,0.07
66,2030,Энергетики,1816.37,forecast,Conservative,0.04


In [142]:
duplicate_check = (
    market_model
    .groupby(["year", "category", "scenario"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_check[
    duplicate_check["count"] > 1
]

print("Duplicate model rows:", len(duplicates))

if len(duplicates) > 0:
    display(duplicates)
else:
    print("✓ No duplicates found")

Duplicate model rows: 0
✓ No duplicates found


In [143]:
historical = market_model[
    market_model["scenario"] == "Historical"
].copy()

base_forecast = market_model[
    market_model["scenario"] == "Base"
].copy()

plot_data = pd.concat([
    historical,
    base_forecast
], ignore_index=True)

fig = px.line(
    plot_data,
    x="year",
    y="volume_mln_l",
    color="category",
    markers=True,
    labels={
        "year": "Год",
        "volume_mln_l": "Объём, млн л",
        "category": "Категория"
    },
    title="Динамика объёма рынка напитков: история и базовый сценарий"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title=None,
    yaxis_title="млн литров",
    legend_title=None
)

fig.show()

# 19. Негазированные напитки

В рамках исследования к негазированным напиткам относятся вода со вкусами,
холодный чай и прочие негазированные прохладительные напитки.

Обычная бутилированная вода и соки исключены из сегмента.

Публичные источники не предоставляют единого показателя, полностью
соответствующего данному определению категории. Поэтому исторический baseline
сегмента является аналитической оценкой и маркируется как `estimate`.

В качестве нижней наблюдаемой границы используется рынок воды со вкусами,
объём которого в 2025 году оценивается примерно в 600 млн литров.
Дополнительно учитываются холодный чай и прочие негазированные категории.

In [144]:
noncarbonated_2025 = 900.0  # analytical estimate, mln liters

# Assume approximately +8% historical growth in 2025
noncarbonated_growth_2025 = 0.08

noncarbonated_2024 = (
    noncarbonated_2025 / (1 + noncarbonated_growth_2025)
)

print(
    f"Non-carbonated drinks 2024 estimate: "
    f"{noncarbonated_2024:.2f} mln l"
)

print(
    f"Non-carbonated drinks 2025 estimate: "
    f"{noncarbonated_2025:.2f} mln l"
)

Non-carbonated drinks 2024 estimate: 833.33 mln l
Non-carbonated drinks 2025 estimate: 900.00 mln l


In [145]:
noncarbonated_baseline = pd.DataFrame([
    {
        "year": 2024,
        "category": "Негазированные",
        "volume_mln_l": noncarbonated_2024,
        "data_type": "estimate"
    },
    {
        "year": 2025,
        "category": "Негазированные",
        "volume_mln_l": noncarbonated_2025,
        "data_type": "estimate"
    }
])

baseline = pd.concat(
    [baseline, noncarbonated_baseline],
    ignore_index=True
)

In [146]:
noncarbonated_scenarios = pd.DataFrame([
    ["Негазированные", "Conservative", 0.01],
    ["Негазированные", "Base",         0.04],
    ["Негазированные", "Optimistic",   0.07],
], columns=[
    "category",
    "scenario",
    "annual_growth"
])

noncarbonated_scenarios["annual_growth_pct"] = (
    noncarbonated_scenarios["annual_growth"] * 100
)

scenario_growth = pd.concat(
    [scenario_growth, noncarbonated_scenarios],
    ignore_index=True
)

display(scenario_growth)

,category,scenario,annual_growth,annual_growth_pct
0,Энергетики,Conservative,0.04,4.0
1,Энергетики,Base,0.07,7.0
2,Энергетики,Optimistic,0.10,10.0
3,Пиво,Conservative,-0.04,-4.0
4,Пиво,Base,-0.02,-2.0
5,Пиво,Optimistic,0.00,0.0
6,Пивные напитки,Conservative,-0.02,-2.0
7,Пивные напитки,Base,0.00,0.0
8,Пивные напитки,Optimistic,0.02,2.0
9,Газированные,Conservative,-0.03,-3.0


In [147]:
forecast_noncarbonated = forecast_category(
    category="Негазированные",
    start_year=2025,
    start_volume=noncarbonated_2025,
    growth_table=scenario_growth
)

forecast = pd.concat(
    [forecast, forecast_noncarbonated],
    ignore_index=True
)

In [148]:
historical_model = baseline.copy()

historical_model["scenario"] = "Historical"
historical_model["annual_growth"] = np.nan

market_model = pd.concat([
    historical_model,
    forecast
], ignore_index=True)

market_model = (
    market_model
    .sort_values(["category", "year", "scenario"])
    .reset_index(drop=True)
)

display(market_model.round(2))

,year,category,volume_mln_l,data_type,scenario,annual_growth
0,2024,Газированные,2971.31,estimate,Historical,NaN
1,2025,Газированные,2900.00,estimate,Historical,NaN
2,2026,Газированные,2871.00,forecast,Base,-0.01
3,2026,Газированные,2813.00,forecast,Conservative,-0.03
4,2026,Газированные,2929.00,forecast,Optimistic,0.01
...,...,...,...,...,...,...
80,2029,Энергетики,1746.51,forecast,Conservative,0.04
81,2029,Энергетики,2185.78,forecast,Optimistic,0.10
82,2030,Энергетики,2093.90,forecast,Base,0.07
83,2030,Энергетики,1816.37,forecast,Conservative,0.04


In [149]:
print("Categories:")
for category in market_model["category"].unique():
    print("-", category)

print(
    "\nNumber of categories:",
    market_model["category"].nunique()
)

Categories:
- Газированные
- Негазированные
- Пивные напитки
- Пиво
- Энергетики

Number of categories: 5


# 20. Оценка рынка в стоимостном выражении

После формирования прогноза натурального объёма рассчитывается стоимостный
объём рынка.

Для каждой категории задаётся средняя розничная цена за литр в базовом
2025 году. В прогнозном периоде цена индексируется с учётом предполагаемого
ежегодного роста цен.

Стоимостный объём рассчитывается по формуле:

\[
MarketValue_t =
\frac{Volume_t \times AveragePrice_t}{1000}
\]

где:

- `Volume` — объём рынка, млн литров;
- `AveragePrice` — средняя цена, руб./л;
- `MarketValue` — объём рынка, млрд руб.

Средние цены используются как модельные предпосылки и отделены от
наблюдаемых исторических данных.

In [150]:
price_assumptions = pd.DataFrame([
    {
        "category": "Энергетики",
        "price_2025_rub_l": 260.0,
        "price_type": "observed",
        "source": "Evotor / market evidence"
    },
    {
        "category": "Газированные",
        "price_2025_rub_l": 101.0,
        "price_type": "observed",
        "source": "Evotor, 2025"
    },
    {
        "category": "Негазированные",
        "price_2025_rub_l": 120.0,
        "price_type": "model_assumption",
        "source": "Analytical assumption"
    },
    {
        "category": "Пиво",
        "price_2025_rub_l": 192.1,
        "price_type": "observed",
        "source": "Rosstat, 2025"
    },
    {
        "category": "Пивные напитки",
        "price_2025_rub_l": 190.0,
        "price_type": "model_assumption",
        "source": "Analytical assumption"
    }
])

display(price_assumptions)

,category,price_2025_rub_l,price_type,source
0,Энергетики,260.0,observed,Evotor / market evidence
1,Газированные,101.0,observed,"Evotor, 2025"
2,Негазированные,120.0,model_assumption,Analytical assumption
3,Пиво,192.1,observed,"Rosstat, 2025"
4,Пивные напитки,190.0,model_assumption,Analytical assumption


# 21. Среднегодовой темп роста рынка

Для оценки долгосрочной динамики используется CAGR
(Compound Annual Growth Rate — среднегодовой темп роста).

\[
CAGR =
\left(
\frac{V_{2030}}{V_{2025}}
\right)^{1/5} - 1
\]

CAGR показывает постоянный среднегодовой темп, который привёл бы рынок
от объёма 2025 года к прогнозируемому объёму 2030 года.

In [151]:
base_2030 = market_model[
    (market_model["year"] == 2030)
    & (market_model["scenario"] == "Base")
][[
    "category",
    "volume_mln_l"
]].rename(
    columns={"volume_mln_l": "volume_2030"}
)

volume_2025 = market_model[
    (market_model["year"] == 2025)
    & (market_model["scenario"] == "Historical")
][[
    "category",
    "volume_mln_l"
]].rename(
    columns={"volume_mln_l": "volume_2025"}
)

cagr_table = volume_2025.merge(
    base_2030,
    on="category",
    how="inner"
)

cagr_table["cagr_2025_2030"] = (
    (cagr_table["volume_2030"] / cagr_table["volume_2025"]) ** (1 / 5)
    - 1
)

cagr_table["cagr_pct"] = (
    cagr_table["cagr_2025_2030"] * 100
)

display(
    cagr_table
    .sort_values("cagr_pct", ascending=False)
    .round(2)
)

,category,volume_2025,volume_2030,cagr_2025_2030,cagr_pct
4,Энергетики,1492.92,2093.90,0.07,7.0
1,Негазированные,900.00,1094.99,0.04,4.0
2,Пивные напитки,955.36,955.36,0.00,0.0
0,Газированные,2900.00,2757.87,-0.01,-1.0
3,Пиво,6070.58,5487.32,-0.02,-2.0


In [152]:
kpi_table = cagr_table.copy()

kpi_table["absolute_change"] = (
    kpi_table["volume_2030"]
    - kpi_table["volume_2025"]
)

kpi_table["change_pct"] = (
    (
        kpi_table["volume_2030"]
        / kpi_table["volume_2025"]
    ) - 1
) * 100

kpi_table = kpi_table[[
    "category",
    "volume_2025",
    "volume_2030",
    "absolute_change",
    "change_pct",
    "cagr_pct"
]]

display(
    kpi_table
    .sort_values("cagr_pct", ascending=False)
    .round(2)
)

,category,volume_2025,volume_2030,absolute_change,change_pct,cagr_pct
4,Энергетики,1492.92,2093.90,600.98,40.26,7.0
1,Негазированные,900.00,1094.99,194.99,21.67,4.0
2,Пивные напитки,955.36,955.36,0.00,0.00,0.0
0,Газированные,2900.00,2757.87,-142.13,-4.90,-1.0
3,Пиво,6070.58,5487.32,-583.26,-9.61,-2.0


In [153]:
fig = px.bar(
    kpi_table.sort_values("cagr_pct"),
    x="cagr_pct",
    y="category",
    orientation="h",
    text_auto=".1f",
    labels={
        "cagr_pct": "CAGR 2025–2030, %",
        "category": ""
    },
    title="Среднегодовая динамика рынка по категориям, 2025–2030"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="CAGR, %",
    yaxis_title=None,
    showlegend=False
)

fig.show()

In [154]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

market_model.to_csv(
    PROCESSED_DIR / "market_model_2024_2030.csv",
    index=False,
    encoding="utf-8-sig"
)

kpi_table.to_csv(
    PROCESSED_DIR / "market_kpi_2025_2030.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ market_model saved")
print("✓ KPI table saved")

✓ market_model saved
✓ KPI table saved


## 22.1. Ценовые предпосылки

Для перевода натурального объёма рынка в стоимостное выражение используются
средние розничные цены за литр.

Для категорий, где доступны публичные показатели, используются наблюдаемые
рыночные цены. При отсутствии полностью сопоставимого показателя применяется
аналитическое ценовое допущение, явно маркированное как `model_assumption`.

Для прогнозного периода предполагается ежегодная индексация средней цены.
Базовый сценарий использует рост цены на 5% в год.

Таким образом, динамика рынка в денежном выражении определяется одновременно:

1. изменением натурального объёма;
2. изменением средней цены.

In [155]:
market_value_2025 = (
    volume_2025
    .merge(
        price_assumptions,
        on="category",
        how="left"
    )
)

market_value_2025["market_value_bln_rub"] = (
    market_value_2025["volume_2025"]
    * market_value_2025["price_2025_rub_l"]
    / 1000
)

display(
    market_value_2025[[
        "category",
        "volume_2025",
        "price_2025_rub_l",
        "market_value_bln_rub",
        "price_type"
    ]]
    .round(2)
)

,category,volume_2025,price_2025_rub_l,market_value_bln_rub,price_type
0,Газированные,2900.00,101.0,292.90,observed
1,Негазированные,900.00,120.0,108.00,model_assumption
2,Пивные напитки,955.36,190.0,181.52,model_assumption
3,Пиво,6070.58,192.1,1166.16,observed
4,Энергетики,1492.92,260.0,388.16,observed


## 22.2. Прогноз стоимостного объёма

Стоимостный прогноз строится на основе натурального Base-сценария.

Средняя цена прогнозируется отдельно:

\[
P_t = P_{t-1}(1+i)
\]

где \(i\) — предполагаемый ежегодный темп роста средней цены.

В базовом сценарии используется ценовая индексация 5% в год.

После этого:

\[
MarketValue_t =
\frac{Volume_t \times Price_t}{1000}
\]

In [156]:
PRICE_GROWTH = 0.05

value_forecast = market_model[
    (
        (market_model["scenario"] == "Base")
        | (market_model["scenario"] == "Historical")
    )
    & (market_model["year"] >= 2025)
].copy()

value_forecast = value_forecast.merge(
    price_assumptions[[
        "category",
        "price_2025_rub_l"
    ]],
    on="category",
    how="left"
)

value_forecast["avg_price_rub_l"] = (
    value_forecast["price_2025_rub_l"]
    * (1 + PRICE_GROWTH) ** (
        value_forecast["year"] - 2025
    )
)

value_forecast["market_value_bln_rub"] = (
    value_forecast["volume_mln_l"]
    * value_forecast["avg_price_rub_l"]
    / 1000
)

display(
    value_forecast[[
        "year",
        "category",
        "volume_mln_l",
        "avg_price_rub_l",
        "market_value_bln_rub"
    ]]
    .sort_values(["category", "year"])
    .round(2)
)

,year,category,volume_mln_l,avg_price_rub_l,market_value_bln_rub
0,2025,Газированные,2900.00,101.00,292.90
1,2026,Газированные,2871.00,106.05,304.47
2,2027,Газированные,2842.29,111.35,316.50
3,2028,Газированные,2813.87,116.92,329.00
4,2029,Газированные,2785.73,122.77,341.99
5,2030,Газированные,2757.87,128.90,355.50
6,2025,Негазированные,900.00,120.00,108.00
7,2026,Негазированные,936.00,126.00,117.94
8,2027,Негазированные,973.44,132.30,128.79
9,2028,Негазированные,1012.38,138.92,140.63


In [157]:
fig = px.line(
    value_forecast,
    x="year",
    y="market_value_bln_rub",
    color="category",
    markers=True,
    labels={
        "year": "Год",
        "market_value_bln_rub": "Объём рынка, млрд ₽",
        "category": "Категория"
    },
    title="Прогноз рынка напитков в стоимостном выражении"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title=None,
    yaxis_title="млрд ₽",
    legend_title=None
)

fig.show()

In [158]:
dashboard_data = value_forecast[[
    "year",
    "category",
    "volume_mln_l",
    "avg_price_rub_l",
    "market_value_bln_rub",
    "data_type",
    "scenario"
]].copy()

dashboard_data = dashboard_data.merge(
    kpi_table[[
        "category",
        "cagr_pct"
    ]],
    on="category",
    how="left"
)

display(dashboard_data.head(10))

,year,category,volume_mln_l,avg_price_rub_l,market_value_bln_rub,data_type,scenario,cagr_pct
0,2025,Газированные,2900.000000,101.000000,292.900000,estimate,Historical,-1.0
1,2026,Газированные,2871.000000,106.050000,304.469550,forecast,Base,-1.0
2,2027,Газированные,2842.290000,111.352500,316.496097,forecast,Base,-1.0
3,2028,Газированные,2813.867100,116.920125,328.997693,forecast,Base,-1.0
4,2029,Газированные,2785.728429,122.766131,341.993102,forecast,Base,-1.0
5,2030,Газированные,2757.871145,128.904438,355.501829,forecast,Base,-1.0
6,2025,Негазированные,900.000000,120.000000,108.000000,estimate,Historical,4.0
7,2026,Негазированные,936.000000,126.000000,117.936000,forecast,Base,4.0
8,2027,Негазированные,973.440000,132.300000,128.786112,forecast,Base,4.0
9,2028,Негазированные,1012.377600,138.915000,140.634434,forecast,Base,4.0


In [159]:
dashboard_data.to_csv(
    PROCESSED_DIR / "dashboard_market_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Dashboard dataset saved")

✓ Dashboard dataset saved
